In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import yaml
import shutil
import torch
import numpy as np

from timeit import default_timer
from torch.utils.data import DataLoader, Subset
from libs.copernicus_h5_dataset import CopernicusH5Dataset, compute_train_uv_stats
from libs.model import Model
from libs.utils import *

def set_random_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
set_random_seed(1234)

In [ ]:
# -------------------------- parameters --------------------------
# read the parameters file
with open(os.path.join('configs', 'IFactFormer.yml'), 'r') as f:
    config = yaml.safe_load(f)
config = dict2namespace(config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------- H5 batch loading --------------------------
h5_path = '/data/copernicus_uv_data/processed_data/uovo_mid_1997-01-01_to_1997-12-31.h5'
dataset = CopernicusH5Dataset(h5_path)
ntrain = int(0.8 * len(dataset))
train_loader = DataLoader(Subset(dataset, range(ntrain)), batch_size=config.training.batch_size, shuffle=True, num_workers=2, persistent_workers=True)
test_loader = DataLoader(Subset(dataset, range(ntrain, len(dataset))), batch_size=config.training.batch_size, shuffle=False, num_workers=2, persistent_workers=True)
print(f'train samples: {ntrain}, test samples: {len(dataset) - ntrain}')
x_batch, y_batch, valid_batch, positions_batch = next(iter(train_loader))
position_shape = tuple(positions_batch['absolute'].shape)
print(f'batch shapes: {tuple(x_batch.shape)} -> {tuple(y_batch.shape)}; mask: {tuple(valid_batch.shape)}; positions: {position_shape}')

以下训练代码已接入海洋掩码和逐切片地理位置。U/V 统计量只来自当前训练窗口的有效海洋像元，mask 不归一化；训练与测试的时间划分仍待改进。

In [ ]:
# create save_dirs
i = 0
save_dirs = os.path.join(config.log_dir, f'dim{config.model.dim}_depth{config.model.depth}_iter{config.model.n_layer}_T{config.model.in_time_window}_{i}')
while os.path.exists(save_dirs):
    i += 1
    save_dirs = os.path.join(config.log_dir, f'dim{config.model.dim}_depth{config.model.depth}_iter{config.model.n_layer}_T{config.model.in_time_window}_{i}')
os.makedirs(save_dirs)
print(save_dirs)

# copy the parameters file
shutil.copyfile('configs/IFactFormer.yml', f'{save_dirs}/IFactFormer.yml')

uv_mean, uv_std = compute_train_uv_stats(dataset, ntrain)
torch.save({'mean': uv_mean, 'std': uv_std, 'train_samples': ntrain, 'input_days': dataset.input_days}, os.path.join(save_dirs, 'uv_stats.pt'))
uv_mean, uv_std = uv_mean.to(device), uv_std.to(device)
print(f'UV mean: {uv_mean.tolist()}, std: {uv_std.tolist()}')

# -------------------------- define --------------------------
model = Model(config.model).to(device)

info = f'count_params: {count_params(model)}'
print(info)
with open(os.path.join(save_dirs, 'training_epoch_log.txt'), 'a') as file:
    file.write(info + '\n')

optimizer = torch.optim.AdamW(model.parameters(), lr=config.training.lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=config.training.scheduler_step, gamma=config.training.scheduler_gamma)

loss_fn = MaskedLpLoss(reduction=False)

train_loss_lst = []
test_loss_lst = []
min_test_loss = 1

# -------------------------- training --------------------------
t0 = default_timer()
for ep in range(config.training.epochs):
    model.train()
    t1 = default_timer()
    for n_iter, (x, y, valid, positions) in enumerate(train_loader):
        x, y, valid = x.to(device), y.to(device), valid.to(device)
        positions = {name: value.to(device) for name, value in positions.items()}
        x = normalize_uv_input(x, uv_mean, uv_std)
        y_pred = model(x, positions) * uv_std + uv_mean
        
        train_loss = torch.mean(loss_fn(y_pred, y, valid))
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()
        
        if n_iter % 100 == 0:
            print(f'train_loss:{train_loss.item():.4f}')
        
    train_loss_lst.append(train_loss.item())
    scheduler.step()  
    
    model.eval()
    with torch.no_grad():
        L2 = []
        for x, y, valid, positions in test_loader:
            x, y, valid = x.to(device), y.to(device), valid.to(device)
            positions = {name: value.to(device) for name, value in positions.items()}
            x = normalize_uv_input(x, uv_mean, uv_std)
            y_pred = model(x, positions) * uv_std + uv_mean
            
            test_L2 = loss_fn(y_pred, y, valid)
            L2.append(test_L2.cpu())
            
        L2 = torch.mean(torch.cat(L2, dim=0), dim=0)
        
    t2 = default_timer()
    
    info = f'{ep} {(t2-t1):.2f} L2:{L2.item():.4f}'
    print(info)
    with open(os.path.join(save_dirs, 'training_epoch_log.txt'), 'a') as file:
        file.write(info + '\n')
        
    if min_test_loss > L2:
        min_test_loss = L2
        torch.save(model.state_dict(), f'{save_dirs}/checkpoint_best.pth')   
    torch.save(model.state_dict(), f'{save_dirs}/checkpoint_{ep}.pth')